# Live Demo — WRDS Data Pull
### Session 5 · ExInt II · WU Vienna · SS 2026

**What we do in this notebook:**
1. Connect to WRDS
2. Explore what Compustat Global looks like
3. Pull 5 firms for fiscal years 2015–2024 — all variables
4. Inspect the data
5. Save as parquet in a timestamped folder

This is the **mini version** of `01_pull_data.py` — same logic, small data so we can see results instantly.

> **Reference project:** https://github.com/vkiefner/sme-intl

---
## Cell 1 — Imports & credentials

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import wrds
import pandas as pd
from datetime import datetime

def find_env():
    current = Path(os.getcwd())
    # Search upward AND in sibling folders at each level
    for path in [current] + list(current.parents):
        # Check current level
        if (path / ".env").exists():
            return path / ".env"
        # Check all siblings at this level
        for sibling in path.iterdir():
            if sibling.is_dir() and (sibling / ".env").exists():
                return sibling / ".env"
    raise FileNotFoundError("Could not find .env anywhere.")

env_file = find_env()
project_root = env_file.parent
os.chdir(project_root)

print(f"Found .env at:     {env_file}")
print(f"Working directory: {os.getcwd()}")

load_dotenv(dotenv_path=env_file, override=True)
WRDS_USER = os.getenv("WRDS_USERNAME")
print(f"WRDS user:         {WRDS_USER}")

Found .env at:     /Users/thomasmacbookair/Documents/medtech-internationalization-drivers/.env
Working directory: /Users/thomasmacbookair/Documents/medtech-internationalization-drivers
WRDS user:         h12015529


---
## Cell 2 — Connect to WRDS

In [10]:
db = wrds.Connection(wrds_username=WRDS_USER)
print("Connected!")

Loading library list...
Done
Connected!


---
## Cell 3 — What tables are available in Compustat Global?

In [3]:
# List all tables in the Compustat Global library
tables = db.list_tables(library="comp_global_daily")
print(f"{len(tables)} tables available:")
for t in tables:
    print(" ", t)

125 tables available:
  dd_group
  dd_group_xref
  dd_item
  dd_package
  g_chars
  g_co_aaudit
  g_co_adesind
  g_co_afnd1
  g_co_afnd2
  g_co_afnddc1
  g_co_afnddc2
  g_co_afntind1
  g_co_afntind2
  g_co_ainvval
  g_co_gsuppl
  g_co_hgic
  g_co_iaudit
  g_co_idesind
  g_co_ifndq
  g_co_ifndsa
  g_co_ifndytd
  g_co_ifntq
  g_co_ifntsa
  g_co_ifntytd
  g_co_industry
  g_co_ipcd
  g_co_offtitl
  g_company
  g_currency
  g_ecind_desc
  g_ecind_mth
  g_exrt_dly
  g_exrt_mth
  g_funda
  g_funda_fncd
  g_fundq
  g_fundq_fncd
  g_idx_daily
  g_idx_index
  g_idx_mth
  g_idxcst_his
  g_indexcst_his
  g_names
  g_names_ix
  g_names_ix_cst
  g_namesq
  g_sec_adesind
  g_sec_adjfact
  g_sec_afnd
  g_sec_afnddc
  g_sec_afnt
  g_sec_divid
  g_sec_dprc
  g_sec_dtrt
  g_sec_gmdivfn
  g_sec_gmth
  g_sec_gmthdiv
  g_sec_gmthprc
  g_sec_history
  g_sec_idesind
  g_sec_ifnd
  g_sec_ifnt
  g_sec_split
  g_secd
  g_secm
  g_secnamesd
  g_security
  g_sedolgvkey
  g_tmptable_pkg6775_tbl5551
  gsecnamesm
  r

---
## Cell 4 — What columns does g_funda have?

In [4]:
# g_funda = Global Fundamentals Annual — the main firm-year table
schema = db.describe_table(library="comp_global_daily", table="g_funda")
print(f"{len(schema)} columns available in g_funda")
schema.head(20)

Approximately 1080399 rows in comp_global_daily.g_funda.
444 columns available in g_funda


,name,nullable,type,comment
0,gvkey,True,VARCHAR(7),Global Company Key
1,indfmt,True,VARCHAR(13),Industry Format
2,datafmt,True,VARCHAR(13),Data Format
3,consol,True,VARCHAR(3),Level of Consolidation - Company Annual Descri...
4,popsrc,True,VARCHAR(2),Population Source
5,acctstd,True,VARCHAR(9),Accounting Standard
6,acqmeth,True,VARCHAR(3),Acquisition Method
7,bspr,True,VARCHAR(9),Balance Sheet Presentation
8,compst,True,VARCHAR(9),Comparability Status
9,curcd,True,VARCHAR(4),ISO Currency Code


---
## Cell 5 — Choose 5 firms to pull

We use `gvkey` — the Compustat firm identifier.  
These are 5 well-known European firms spanning different countries and sectors.

| gvkey  | Firm     | Country |
|--------|----------|---------|
| 100737 | Volkswagen  | DEU     |
| 016603 | Nestlé   | CHE     |
| 024625 | TotalEnergies   | FRA     |
| 014447 | LVMH  | FRA     |
| 017436 | BASF | DEU     |

In [6]:
# Real verified gvkeys from Compustat Global
# Siemens Healthineers AG (DEU), Sartorius AG (DEU), Carl Zeiss Meditec AG (DEU), Smith & Nephew plc (GBR), Getinge AB (SWE)
FIVE_FIRMS = ('326765', '208821', '238442', '101317', '221806')

firms_sql = ", ".join(f"'{g}'" for g in FIVE_FIRMS)
print(f"Pulling: Siemens Healthineers AG, Sartorius AG, Carl Zeiss Meditec AG, Smith & Nephew plc, Getinge AB")
print(f"gvkeys:  {firms_sql}")

Pulling: Siemens Healthineers AG, Sartorius AG, Carl Zeiss Meditec AG, Smith & Nephew plc, Getinge AB
gvkeys:  '326765', '208821', '238442', '101317', '221806'


---
## Cell 6 — Pull ALL variables for these 5 firms, 2015–2024

In [11]:
# Step 1: try without any filters first to confirm the firms exist
query = f"""
    SELECT *
    FROM comp_global_daily.g_funda
    WHERE gvkey   IN ({firms_sql})
      AND fyear   BETWEEN 2015 AND 2025
      AND datafmt = 'HIST_STD'
      AND indfmt  = 'INDL'
      AND popsrc  = 'I'
      AND consol  = 'C'
    ORDER BY gvkey, fyear
"""

print("Pulling from WRDS...")
df_raw = db.raw_sql(query, date_cols=["datadate"])
print(f"\nDone! Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

Pulling from WRDS...

Done! Shape: 55 rows × 444 columns


---
## Cell 7 — First look at the data

In [12]:
# First 5 rows
df_raw.head()

,gvkey,indfmt,datafmt,consol,popsrc,acctstd,acqmeth,bspr,compst,curcd,...,conm,costat,fic,loc,naicsh,sich,rank,au,auop,hiid
0,101317,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,USD,...,SMITH & NEPHEW PLC,A,GBR,GBR,339113,3842,1,6,1,01W
1,101317,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,USD,...,SMITH & NEPHEW PLC,A,GBR,GBR,339113,3842,1,6,1,01W
2,101317,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,USD,...,SMITH & NEPHEW PLC,A,GBR,GBR,339113,3842,1,6,1,01W
3,101317,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,USD,...,SMITH & NEPHEW PLC,A,GBR,GBR,339113,3842,1,6,1,01W
4,101317,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,USD,...,SMITH & NEPHEW PLC,A,GBR,GBR,339113,3842,1,6,4,01W


In [13]:
# Column names — all available variables
print(f"Total columns: {len(df_raw.columns)}")
print("\nAll column names:")
for i, col in enumerate(df_raw.columns):
    print(f"  {i:>3}. {col}")

Total columns: 444

All column names:
    0. gvkey
    1. indfmt
    2. datafmt
    3. consol
    4. popsrc
    5. acctstd
    6. acqmeth
    7. bspr
    8. compst
    9. curcd
   10. final
   11. fyear
   12. fyr
   13. ismod
   14. pddur
   15. scf
   16. src
   17. stalt
   18. upd
   19. datadate
   20. fdate
   21. pdate
   22. accli
   23. acco
   24. aco
   25. acofs
   26. acox
   27. acoxfs
   28. acqdisn
   29. acqdiso
   30. act
   31. adpac
   32. am
   33. amdc
   34. ao
   35. aoloch
   36. aox
   37. ap
   38. apalch
   39. apch
   40. apdpfs
   41. apfs
   42. apo
   43. apofs
   44. aqc
   45. artfs
   46. asdis
   47. asinv
   48. at
   49. atoch
   50. autxr
   51. bcef
   52. bct
   53. ca
   54. capcst
   55. capfl
   56. capr1
   57. capr2
   58. capr3
   59. caprt
   60. caps
   61. capx
   62. capxfi
   63. ceq
   64. cfbd
   65. cfere
   66. cflaoth
   67. cfo
   68. cfpdo
   69. cga
   70. ch
   71. che
   72. cheb
   73. chech
   74. chee
   75. chefs
   76. 

In [14]:
# Data types
df_raw.dtypes

gvkey      string[python]
indfmt     string[python]
datafmt    string[python]
consol     string[python]
popsrc     string[python]
                ...      
sich                Int64
rank                Int64
au         string[python]
auop       string[python]
hiid       string[python]
Length: 444, dtype: object

In [15]:
# Which firms did we get? How many years each?
df_raw.groupby(["gvkey", "conm"])["fyear"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_year", "max": "last_year", "count": "n_years"}
)

,,first_year,last_year,n_years
gvkey,conm,,,
101317,SMITH & NEPHEW PLC,2015,2025,11
208821,SARTORIUS AG,2015,2025,11
221806,GETINGE AB PUBL,2015,2025,11
238442,CARL ZEISS MEDITEC AG,2015,2025,11
326765,SIEMENS HEALTHINEE,2015,2025,11


---
## Cell 8 — Look at key financial variables

In [18]:
# Select the most commonly used variables for a quick preview
key_vars = ["gvkey", "conm", "fyear", "loc",
            "at",    # total assets
            "sale",  # net sales
            "ib",    # income before extraordinary items (net income)
            "xrd",   # R&D expenditure
            "emp",   # employees (thousands)
            "dltt",  # long-term debt
            "dlc",  # debt in current liabilities (Leverage)
            "seq",  # stakeholder equity (Leverage)
            "sich", # SIC code historical (Branche)
            "ebitda", # EBITDA (Performance-Alternative)
            "naics", # NAICS Branchen-Klassifikation (Robustheit)

            "curcd" # currency
           ]

# Keep only columns that exist in our pull
available = [v for v in key_vars if v in df_raw.columns]
df_raw[available]

,gvkey,conm,fyear,loc,at,sale,ib,xrd,emp,dltt,dlc,seq,sich,ebitda,curcd
0,101317,SMITH & NEPHEW PLC,2015,GBR,7167.0,4634.0,410.0,222.0,15.644,1434.0,46.0,3966.0,3842,1381.0,USD
1,101317,SMITH & NEPHEW PLC,2016,GBR,7344.0,4669.0,784.0,278.0,15.644,1564.0,86.0,3958.0,3842,1327.0,USD
2,101317,SMITH & NEPHEW PLC,2017,GBR,7866.0,4765.0,767.0,229.0,15.933,1423.0,27.0,4644.0,3842,1320.0,USD
3,101317,SMITH & NEPHEW PLC,2018,GBR,8059.0,4904.0,663.0,270.0,16.377,1301.0,164.0,4874.0,3842,1396.0,USD
4,101317,SMITH & NEPHEW PLC,2019,GBR,9299.0,5138.0,600.0,320.0,17.637,1975.0,72.0,5141.0,3842,1477.0,USD
5,101317,SMITH & NEPHEW PLC,2020,GBR,11012.0,4560.0,448.0,344.0,17.914,3353.0,337.0,5279.0,3842,978.0,USD
6,101317,SMITH & NEPHEW PLC,2021,GBR,10920.0,5212.0,524.0,397.0,18.369,2848.0,491.0,5568.0,3842,1276.0,USD
7,101317,SMITH & NEPHEW PLC,2022,GBR,9966.0,5215.0,223.0,396.0,19.012,2712.0,160.0,5259.0,3842,1201.0,USD
8,101317,SMITH & NEPHEW PLC,2023,GBR,9987.0,5549.0,263.0,385.0,18.452,2319.0,765.0,5217.0,3842,1265.0,USD
9,101317,SMITH & NEPHEW PLC,2024,GBR,10354.0,5810.0,412.0,343.0,17.349,3258.0,63.0,5265.0,3842,1441.0,USD


In [19]:
# Quick summary stats on numeric columns
df_raw[["at", "sale", "ib", "xrd", "emp","dltt", "dlc", "seq", "ebitda"]].describe().round(2)

,at,sale,ib,xrd,emp,dltt,dlc,seq,ebitda
count,55.0,55.0,55.0,55.0,53.0,55.0,55.0,55.0,55.0
mean,19891.23,11112.37,843.4,759.69,19.52,4060.95,1394.15,8806.32,2098.94
std,19831.86,10993.59,867.46,687.34,20.15,4373.6,1956.64,9466.94,1934.12
min,1139.29,1040.06,-967.0,59.13,2.89,3.0,2.82,517.73,146.31
25%,3213.06,2077.71,158.25,207.5,7.5,702.17,54.5,1824.51,427.74
50%,9987.0,5138.0,524.0,373.56,12.83,2848.0,337.46,4644.0,1327.0
75%,42757.0,21697.0,1400.5,1333.0,17.64,5106.5,2483.0,18060.5,3449.0
max,63918.0,34969.0,3239.0,2160.0,72.0,16006.0,8492.0,33005.0,6824.0


---
## Cell 9 — How many missing values per column?

In [20]:
# Missing value count and percentage
missing = pd.DataFrame({
    "missing_n":   df_raw.isnull().sum(),
    "missing_pct": (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
})

# Show only columns that have at least one missing value
missing[missing["missing_n"] > 0].sort_values("missing_pct", ascending=False)

,missing_n,missing_pct
acqmeth,55,100.0
oancfd,55,100.0
pliach,55,100.0
pcl,55,100.0
pacqp,55,100.0
...,...,...
invch,3,5.5
emp,2,3.6
exre,1,1.8
txop,1,1.8


---
## Cell 10 — Standardize column names

In [21]:
# Lowercase + strip whitespace — same as the full pull script
df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

print("Column names standardized.")
print("Before:", list(df_raw.columns[:5]))
print("After: ", list(df.columns[:5]))

Column names standardized.
Before: ['gvkey', 'indfmt', 'datafmt', 'consol', 'popsrc']
After:  ['gvkey', 'indfmt', 'datafmt', 'consol', 'popsrc']


---
## Cell 11 — Save to timestamped parquet folder

Same structure as `01_pull_data.py`: each run gets its own folder named with the current date and time.

In [22]:
# Create timestamped output folder
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir   = Path("data") / "raw" / timestamp
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {out_dir}")

# Save one parquet file per year (same chunking strategy as full pull)
for year, group in df.groupby("fyear"):
    out_path = out_dir / f"fyear_{int(year)}.parquet"
    group.to_parquet(out_path, index=False)
    print(f"  fyear_{int(year)}.parquet  →  {len(group)} rows")

# Write metadata
(out_dir / "pull_metadata.txt").write_text(
    f"LIVE DEMO PULL (5 firms)\n"
    f"========================\n"
    f"Pulled:       {datetime.now().isoformat()}\n"
    f"Source:       comp_global_daily.g_funda\n"
    f"Firms:        {', '.join(FIVE_FIRMS)}\n"
    f"Fiscal years: 2015–2024\n"
    f"Rows:         {len(df)}\n"
    f"Columns:      {df.shape[1]}\n"
)

print(f"\nDone! {len(df)} rows saved across {df['fyear'].nunique()} year files.")

Output folder: data/raw/2026-05-29_18-35-39
  fyear_2015.parquet  →  5 rows
  fyear_2016.parquet  →  5 rows
  fyear_2017.parquet  →  5 rows
  fyear_2018.parquet  →  5 rows
  fyear_2019.parquet  →  5 rows
  fyear_2020.parquet  →  5 rows
  fyear_2021.parquet  →  5 rows
  fyear_2022.parquet  →  5 rows
  fyear_2023.parquet  →  5 rows
  fyear_2024.parquet  →  5 rows
  fyear_2025.parquet  →  5 rows

Done! 55 rows saved across 11 year files.


---
## Cell 12 — Verify: read back one parquet file

In [23]:
# Read back the 2025 file to confirm it saved correctly
check = pd.read_parquet(out_dir / "fyear_2025.parquet")
print(f"fyear_2025.parquet: {check.shape[0]} rows × {check.shape[1]} columns")
check[["gvkey", "conm", "fyear", "at", "sale", "ib"]].head()

fyear_2025.parquet: 5 rows × 444 columns


,gvkey,conm,fyear,at,sale,ib
0,101317,SMITH & NEPHEW PLC,2025,10457.0,6164.0,625.0
1,208821,SARTORIUS AG,2025,9717.2,3538.1,154.9
2,221806,GETINGE AB PUBL,2025,56505.0,34969.0,2258.0
3,238442,CARL ZEISS MEDITEC AG,2025,3403.371,2227.645,141.21
4,326765,SIEMENS HEALTHINEE,2025,44370.0,23375.0,2144.0


---
## Cell 13 — Close the connection

In [24]:
db.close()
print("WRDS connection closed.")
print(f"\nYour data is in: {out_dir}")
print(f"Next step:       run 02_clean.py (or the full 01_pull_data.py for your own project)")

WRDS connection closed.

Your data is in: data/raw/2026-05-29_18-35-39
Next step:       run 02_clean.py (or the full 01_pull_data.py for your own project)


---
## Summary — What just happened

| Step | What we did |
|------|-------------|
| Connected | `wrds.Connection()` using credentials from `.env` |
| Explored | Listed tables and columns available in Compustat Global |
| Queried | `SELECT *` for 5 firms, 2015–2024, with standard filters |
| Inspected | Shape, column names, dtypes, missing values |
| Cleaned names | Lowercase column names |
| Saved | Parquet files in a timestamped folder under `data/raw/` |

**Your full script `01_pull_data.py` does exactly the same thing** — just for ALL firms globally instead of 5, and chunked by fiscal year to handle the larger volume.

---
*ExInt II · WU Vienna · SS 2026 · github.com/vkiefner/sme-intl*